# Agent understanding: feature store and RAG

This notebook shows one specialist agent calling the governed feature/RAG MCP server. The request is scoped to one ticker and one RAG chunk; the tool returns structured features, retrieved context, and source metadata. Set the endpoint and fixture identifiers before executing.

In [1]:
import asyncio
import json
import os

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

FEATURE_MCP_URL = os.environ.get("FEATURE_MCP_URL", "http://feature-mcp.phase2-data.svc.cluster.local/mcp")
USER_ID = os.environ.get("DEMO_USER_ID", "VNM")
CHUNK_ID = os.environ.get("DEMO_CHUNK_ID", "c1")
SCOPE = os.environ.get("DEMO_SCOPE", "financial-distress:read")
AGENT_IDENTITY = os.environ.get("DEMO_AGENT_IDENTITY", "feature-agent")
FEATURE_NAMES = tuple(os.environ.get("DEMO_FEATURE_NAMES", "company_risk_features:z_score").split(","))

agent_understanding = {"identity": AGENT_IDENTITY, "scope": SCOPE, "tool": "lookup_feature_context"}
agent_understanding

{'identity': 'feature-agent', 'scope': 'financial-distress:read', 'tool': 'lookup_feature_context'}

In [2]:
async def call_feature_mcp() -> dict:
    request = {
        "agent_identity": AGENT_IDENTITY,
        "scope": SCOPE,
        "user_id": USER_ID,
        "feature_names": list(FEATURE_NAMES),
        "chunk_id": CHUNK_ID,
    }
    async with streamable_http_client(FEATURE_MCP_URL) as (read, write, _) :
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("lookup_feature_context", {"request": request})
            return result.structuredContent or {"content": [item.model_dump() for item in result.content]}

feature_result = asyncio.run(call_feature_mcp())
print(json.dumps(feature_result, indent=2, default=str))

{
  "ok": true,
  "data": {
    "features": {
      "z_score": null
    },
    "rag": {
      "chunk_id": "phase3-chunk",
      "chunk_text": "Audited Phase 03 evidence chunk.",
      "source_uri": "https://example.com/phase3",
      "company": "VNM",
      "report_date": "2026-08-10",
      "access_class": "public"
    }
  },
  "error": null
}


## Interpretation

The agent does not connect to Redis or PostgreSQL directly. The MCP tool validates identity and scope, performs the online feature lookup and RAG chunk lookup, and returns source metadata that a coordinator can cite.